# 第7章 学習とは坂を下ること：微分と勾配降下法

目隠しで山を下りる方法と、PyTorchの自動微分

教科書のページ：https://sakunokyumei.github.io/llm-research-textbook/07-gradient.html

上のセルから順に、再生ボタン（▶）か `Shift + Enter` で実行します。説明は教科書のページにあります。


## ボンゴ焼き問題


In [ ]:
hours = [1, 2, 3, 4, 5]          # 焼いた時間
sold = [2.9, 6.2, 8.8, 12.1, 15.0]  # 売れた個数（10個単位）

In [ ]:
def predict(w, x):
    return w * x

def loss_fn(w):
    total = 0.0
    for x, y in zip(hours, sold):
        total += (predict(w, x) - y) ** 2
    return total / len(hours)

for w in [1, 2, 3, 4]:
    print(f"w = {w} のとき、損失 = {loss_fn(w):.3f}")

In [ ]:
!pip install -q matplotlib-fontja

In [ ]:
import matplotlib.pyplot as plt
import matplotlib_fontja

ws = [i / 10 for i in range(0, 61)]
plt.figure(figsize=(6, 3.5))
plt.plot(ws, [loss_fn(w) for w in ws])
plt.xlabel("パラメータ w")
plt.ylabel("損失")
plt.title("損失の「谷」：いちばん低い場所の w が答え")
plt.grid(alpha=0.3)
plt.show()

## 目隠しで山を下りる


In [ ]:
def slope(w, h=0.0001):
    return (loss_fn(w + h) - loss_fn(w)) / h

for w in [1, 2, 3, 4]:
    print(f"w = {w} での傾き：{slope(w):8.3f}")

## 勾配降下法


In [ ]:
w = 0.0
lr = 0.01
for step in range(15):
    w = w - lr * slope(w)
    if step % 2 == 0:
        print(f"ステップ {step:2d}：w = {w:.4f}  損失 = {loss_fn(w):.4f}")

## パラメータが2つになったら：勾配


In [ ]:
def loss2(w, b):
    return sum((w * x + b - y) ** 2 for x, y in zip(hours, sold)) / len(hours)

def slopes(w, b, h=0.0001):
    dw = (loss2(w + h, b) - loss2(w, b)) / h   # w だけ少し動かす
    db = (loss2(w, b + h) - loss2(w, b)) / h   # b だけ少し動かす
    return dw, db

w, b = 0.0, 0.0
for step in range(2000):
    dw, db = slopes(w, b)
    w, b = w - 0.02 * dw, b - 0.02 * db
print(f"w = {w:.3f}, b = {b:.3f}, 損失 = {loss2(w, b):.4f}")

## PyTorchの自動微分


In [ ]:
import torch

x = torch.tensor(hours, dtype=torch.float32)
y = torch.tensor(sold)

w = torch.tensor(1.0, requires_grad=True)   # 「この数の勾配を知りたい」と宣言
loss = ((w * x - y) ** 2).mean()
loss.backward()                              # 勾配を計算！
print("損失：", loss.item())
print("自動微分の傾き：", w.grad.item())
print("数値微分の傾き：", slope(1.0))

In [ ]:
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
lr = 0.02

for step in range(2000):
    loss = ((w * x + b - y) ** 2).mean()   # ① 予想して、損失を計算
    loss.backward()                         # ② 勾配を計算
    with torch.no_grad():                   # ③ パラメータを更新（ここは記録しない）
        w -= lr * w.grad
        b -= lr * b.grad
    w.grad.zero_()                          # ④ 勾配を0に戻す
    b.grad.zero_()

print(f"w = {w.item():.3f}, b = {b.item():.3f}, 損失 = {loss.item():.4f}")

## 自動微分の中身：連鎖律


In [ ]:
x2 = torch.tensor(2.0, requires_grad=True)   # 上のデータの x と区別するため、x2 という名前に
u = 3 * x2 + 1
y2 = u ** 2
y2.backward()
print("自動微分：", x2.grad.item())
print("連鎖律で手計算：", 2 * u.item() * 3)